# Cortex Agent Cost Observability

**Duration:** 30 minutes  
**Scenario:** You're a FinOps admin responsible for understanding and managing the cost of Cortex Agent deployments. You need to answer: how many tokens are being consumed, who is consuming them, how much do Analyst tool calls cost, and what warehouse compute is triggered by agent queries?

**What you'll learn:**
1. Query per-request token and credit consumption from `CORTEX_AGENT_USAGE_HISTORY`
2. Break down costs by model, service type (agent orchestration vs analyst), and token category (input/output/cache)
3. Identify top consumers by user and agent
4. Measure Cortex Analyst credits from `CORTEX_ANALYST_USAGE_HISTORY`
5. Attribute warehouse compute costs via `QUERY_ATTRIBUTION_HISTORY`
6. Calculate total cost per request (tokens + compute)

**Prerequisites:** Run `setup.sql` before starting. It creates the database, warehouse, and simulated fallback data.

**Documentation:**
- [CORTEX_AGENT_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_agent_usage_history)
- [CORTEX_ANALYST_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history)
- [QUERY_ATTRIBUTION_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history)
- [Resource Budgets for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-resource-budgets)

In [1]:
# Connection setup — works in Snowsight notebooks and local Jupyter via ~/.snowflake/config.toml
import pandas as pd

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local: read connection from ~/.snowflake/config.toml
    from snowflake.snowpark import Session
    import os, tomllib
    from pathlib import Path

    config_path = Path.home() / ".snowflake" / "config.toml"
    with open(config_path, "rb") as f:
        config = tomllib.load(f)

    # Use SNOWFLAKE_DEFAULT_CONNECTION_NAME env var, or the default_connection_name from config
    conn_name = os.environ.get(
        "SNOWFLAKE_DEFAULT_CONNECTION_NAME",
        config.get("default_connection_name", "default")
    )
    conn_params = config["connections"][conn_name]

    session = Session.builder.configs(conn_params).create()

# Set context
session.sql("USE DATABASE AGENT_COST_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE AGENT_COST_LAB_WH").collect()

print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

Connected as: PERICKSON
Role: ACCOUNTADMIN


In [2]:
# Detect whether live ACCOUNT_USAGE data is available
try:
    test = session.sql("""
        SELECT COUNT(*) AS cnt 
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    """).collect()
    USE_LIVE_DATA = test[0][0] > 0
except Exception:
    USE_LIVE_DATA = False

if USE_LIVE_DATA:
    print("Live data detected in CORTEX_AGENT_USAGE_HISTORY. Using production views.")
else:
    print("No live agent data found (or insufficient privileges). Using simulated data.")
    print("Tip: Run setup.sql first, and ensure your role has access to SNOWFLAKE.ACCOUNT_USAGE.")

Live data detected in CORTEX_AGENT_USAGE_HISTORY. Using production views.


---
## Section 1: Aggregate Token and Credit Consumption

The `CORTEX_AGENT_USAGE_HISTORY` view provides per-request granularity with:
- **TOKENS**: Total tokens consumed by the agent request
- **TOKEN_CREDITS**: Credits charged for those tokens

Each row = one agent interaction (message request/response pair).

In [3]:
# Aggregate token and credit consumption (last 30 days)
if USE_LIVE_DATA:
    df_agg = session.sql("""
        SELECT 
            COUNT(*) AS total_requests,
            SUM(TOKENS) AS total_tokens,
            SUM(TOKEN_CREDITS) AS total_credits,
            AVG(TOKENS) AS avg_tokens_per_request,
            AVG(TOKEN_CREDITS) AS avg_credits_per_request
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    """).to_pandas()
else:
    df_agg = session.sql("""
        SELECT 
            COUNT(DISTINCT REQUEST_ID) AS total_requests,
            SUM(INPUT_TOKENS + OUTPUT_TOKENS) AS total_tokens,
            SUM(TOKEN_CREDITS) AS total_credits,
            AVG(INPUT_TOKENS + OUTPUT_TOKENS) AS avg_tokens_per_request,
            AVG(TOKEN_CREDITS) AS avg_credits_per_request
        FROM SIMULATED_AGENT_USAGE
    """).to_pandas()

print("=== Aggregate Agent Token Consumption ===")
print(df_agg.to_string(index=False))

=== Aggregate Agent Token Consumption ===
 TOTAL_REQUESTS  TOTAL_TOKENS  TOTAL_CREDITS  AVG_TOKENS_PER_REQUEST  AVG_CREDITS_PER_REQUEST
            725      58483013      72.977595            80666.224828                 0.100659


---
## Section 2: Per-Model Token Breakdown (TOKENS_GRANULAR)

The `TOKENS_GRANULAR` array column contains a per-sub-request breakdown:
- **service_type**: `cortex_agents` (orchestration) vs `cortex_analyst` (tool calls)
- **model**: Which LLM model was used (e.g., claude-4-sonnet, llama-4-maverick)
- **Token categories**: `input`, `output`, `cache_read_input`, `cache_write_input`

Cache read tokens are cheaper -- a high cache hit ratio means lower effective cost.

In [4]:
# Per-model token breakdown with cache hit ratio
if USE_LIVE_DATA:
    df_model = session.sql("""
        WITH flattened AS (
            SELECT 
                tg.key AS sub_request_id,
                st.key AS service_type,
                m.key AS model_name,
                m.value:input::NUMBER AS input_tokens,
                m.value:cache_read_input::NUMBER AS cache_read_tokens,
                m.value:cache_write_input::NUMBER AS cache_write_tokens,
                m.value:output::NUMBER AS output_tokens
            FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
                LATERAL FLATTEN(input => TOKENS_GRANULAR) tg_outer,
                LATERAL FLATTEN(input => tg_outer.value) tg,
                LATERAL FLATTEN(input => tg.value) st,
                LATERAL FLATTEN(input => st.value) m
            WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
                AND tg.key != 'start_time'
                AND st.key != 'start_time'
        )
        SELECT 
            service_type,
            model_name,
            SUM(input_tokens) AS total_input,
            SUM(output_tokens) AS total_output,
            SUM(cache_read_tokens) AS total_cache_read,
            ROUND(SUM(cache_read_tokens) / NULLIF(SUM(input_tokens), 0) * 100, 1) AS cache_hit_pct
        FROM flattened
        WHERE model_name NOT IN ('start_time')
        GROUP BY 1, 2
        ORDER BY total_input DESC
    """).to_pandas()
else:
    df_model = session.sql("""
        SELECT 
            SERVICE_TYPE,
            MODEL_NAME,
            SUM(INPUT_TOKENS) AS TOTAL_INPUT,
            SUM(OUTPUT_TOKENS) AS TOTAL_OUTPUT,
            SUM(CACHE_READ_INPUT_TOKENS) AS TOTAL_CACHE_READ,
            ROUND(SUM(CACHE_READ_INPUT_TOKENS) / NULLIF(SUM(INPUT_TOKENS), 0) * 100, 1) AS CACHE_HIT_PCT
        FROM SIMULATED_AGENT_USAGE
        GROUP BY 1, 2
        ORDER BY TOTAL_INPUT DESC
    """).to_pandas()

print("=== Token Breakdown by Service Type and Model ===")
print(df_model.to_string(index=False))
print("\nInsight: cache_hit_pct > 0 means prompt-cache reuse is reducing effective cost.")

=== Token Breakdown by Service Type and Model ===
 SERVICE_TYPE        MODEL_NAME  TOTAL_INPUT  TOTAL_OUTPUT  TOTAL_CACHE_READ  CACHE_HIT_PCT
cortex_agents   claude-opus-4-6       305442        114873           2882160          943.6
cortex_agents claude-sonnet-4-5         6337        878862          20952582       330638.8
cortex_agents   claude-opus-4-7         2584        252216          10246686       396543.6
cortex_agents   claude-opus-4-5          612        311388           9680417      1581767.5
cortex_agents   claude-opus-4-8          234         31680           1223116       522699.1

Insight: cache_hit_pct > 0 means prompt-cache reuse is reducing effective cost.


---
## Section 3: Per-User Consumption Leaderboard

Who is consuming the most credits? Group by `USER_NAME` and `AGENT_NAME` to identify top consumers for chargeback or governance.

In [5]:
# Per-user consumption leaderboard
if USE_LIVE_DATA:
    df_users = session.sql("""
        SELECT 
            USER_NAME,
            AGENT_NAME,
            COUNT(*) AS request_count,
            SUM(TOKENS) AS total_tokens,
            SUM(TOKEN_CREDITS) AS total_credits
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1, 2
        ORDER BY total_credits DESC
        LIMIT 15
    """).to_pandas()
else:
    df_users = session.sql("""
        SELECT 
            USER_NAME,
            AGENT_NAME,
            COUNT(DISTINCT REQUEST_ID) AS REQUEST_COUNT,
            SUM(INPUT_TOKENS + OUTPUT_TOKENS) AS TOTAL_TOKENS,
            SUM(TOKEN_CREDITS) AS TOTAL_CREDITS
        FROM SIMULATED_AGENT_USAGE
        GROUP BY 1, 2
        ORDER BY TOTAL_CREDITS DESC
        LIMIT 15
    """).to_pandas()

print("=== Top Consumers by User + Agent ===")
print(df_users.to_string(index=False))

=== Top Consumers by User + Agent ===
USER_NAME         AGENT_NAME  REQUEST_COUNT  TOTAL_TOKENS  TOTAL_CREDITS
     None           SALESBRO            450      43126302      48.373268
     None           LOYALBRO             97       7235675       9.642008
PERICKSON      CMO_ASSISTANT             28       1348604       2.812028
     None      CMO_ASSISTANT             24       1850192       2.491244
PERICKSON       BITEIQ_AGENT             17       1260318       1.867431
PERICKSON           LOYALBRO             13       1287093       1.776025
PERICKSON         SUPERVISOR             40        424418       1.761648
PERICKSON      SALES_ANALYST             16        513337       1.035769
PERICKSON TENANT_SALES_AGENT              9        447643       0.895504
     None         SUPERVISOR             12        145548       0.627364
PERICKSON       SUPPLY_CHAIN              5        190087       0.473612
PERICKSON    PRODUCT_SUPPORT              5        117233       0.253511
     None    

---
## Section 4: Cortex Analyst Credits

When a Cortex Agent uses Cortex Analyst as a tool, those credits are tracked in two places:
1. **Within `CORTEX_AGENT_USAGE_HISTORY`**: The `TOKENS_GRANULAR` array has entries with `service_type = 'cortex_analyst'`
2. **Separately in `CORTEX_ANALYST_USAGE_HISTORY`**: Hourly rollup of analyst credits by username

The `METADATA:ai_functions_credits` field in `CORTEX_AGENT_USAGE_HISTORY` also captures credits from any AI functions invoked during the request.

In [6]:
# Cortex Analyst usage: hourly credits by user
if USE_LIVE_DATA:
    df_analyst = session.sql("""
        SELECT 
            USERNAME,
            SUM(REQUEST_COUNT) AS total_requests,
            SUM(CREDITS) AS total_analyst_credits
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
        WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1
        ORDER BY total_analyst_credits DESC
    """).to_pandas()
else:
    df_analyst = session.sql("""
        SELECT 
            USERNAME,
            SUM(REQUEST_COUNT) AS TOTAL_REQUESTS,
            SUM(CREDITS) AS TOTAL_ANALYST_CREDITS
        FROM SIMULATED_ANALYST_USAGE
        GROUP BY 1
        ORDER BY TOTAL_ANALYST_CREDITS DESC
    """).to_pandas()

print("=== Cortex Analyst Credits by User ===")
print(df_analyst.to_string(index=False))
print("\nNote: These credits roll into the AI_SERVICES service type in METERING_DAILY_HISTORY.")

=== Cortex Analyst Credits by User ===
Empty DataFrame
Columns: [USERNAME, TOTAL_REQUESTS, TOTAL_ANALYST_CREDITS]
Index: []

Note: These credits roll into the AI_SERVICES service type in METERING_DAILY_HISTORY.


In [7]:
# Analyst-specific token breakdown from CORTEX_AGENT_USAGE_HISTORY
if USE_LIVE_DATA:
    print("In production, filter TOKENS_GRANULAR for service_type = 'cortex_analyst':")
    print("""
    -- Extract analyst-specific costs from the agent usage view
    SELECT 
        USER_NAME,
        AGENT_NAME,
        f.value:service_type::STRING AS service_type,
        f.value:model::STRING AS model,
        SUM(f.value:input::NUMBER) AS analyst_input_tokens,
        SUM(f.value:output::NUMBER) AS analyst_output_tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
        LATERAL FLATTEN(input => TOKENS_GRANULAR) f_outer,
        LATERAL FLATTEN(input => f_outer.value) f
    WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        AND f.value:service_type::STRING = 'cortex_analyst'
    GROUP BY 1, 2, 3, 4
    ORDER BY analyst_input_tokens DESC;
    """)
else:
    df_analyst_detail = session.sql("""
        SELECT 
            USER_NAME,
            AGENT_NAME,
            MODEL_NAME,
            SUM(INPUT_TOKENS) AS ANALYST_INPUT_TOKENS,
            SUM(OUTPUT_TOKENS) AS ANALYST_OUTPUT_TOKENS,
            SUM(TOKEN_CREDITS) AS ANALYST_CREDITS
        FROM SIMULATED_AGENT_USAGE
        WHERE SERVICE_TYPE = 'cortex_analyst'
        GROUP BY 1, 2, 3
        ORDER BY ANALYST_CREDITS DESC
    """).to_pandas()
    print("=== Analyst Token Detail (from Agent Usage) ===")
    print(df_analyst_detail.to_string(index=False))

In production, filter TOKENS_GRANULAR for service_type = 'cortex_analyst':

    -- Extract analyst-specific costs from the agent usage view
    SELECT 
        USER_NAME,
        AGENT_NAME,
        f.value:service_type::STRING AS service_type,
        f.value:model::STRING AS model,
        SUM(f.value:input::NUMBER) AS analyst_input_tokens,
        SUM(f.value:output::NUMBER) AS analyst_output_tokens
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
        LATERAL FLATTEN(input => TOKENS_GRANULAR) f_outer,
        LATERAL FLATTEN(input => f_outer.value) f
    WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        AND f.value:service_type::STRING = 'cortex_analyst'
    GROUP BY 1, 2, 3, 4
    ORDER BY analyst_input_tokens DESC;
    


---
## Section 5: Warehouse Compute Attribution

Token credits are only part of the cost. When an agent invokes Cortex Analyst, the generated SQL runs on a warehouse. That warehouse compute is tracked in `QUERY_ATTRIBUTION_HISTORY`.

Key facts:
- `CREDITS_ATTRIBUTED_COMPUTE` = warehouse credits for executing the query (excludes idle time)
- Latency: up to 8 hours for data to appear
- Short queries (< ~100ms) may not appear in this view
- Use `QUERY_TAG` on your sessions to make attribution easier

In [8]:
# Warehouse compute attribution for agent-triggered queries
if USE_LIVE_DATA:
    df_compute = session.sql("""
        SELECT 
            a.USER_NAME,
            a.WAREHOUSE_NAME,
            COUNT(*) AS query_count,
            SUM(a.CREDITS_ATTRIBUTED_COMPUTE) AS total_warehouse_credits
        FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY a
        WHERE a.START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
            AND (a.QUERY_TAG ILIKE '%agent%' OR a.QUERY_TAG ILIKE '%cortex%')
        GROUP BY 1, 2
        ORDER BY total_warehouse_credits DESC
        LIMIT 10
    """).to_pandas()
else:
    df_compute = session.sql("""
        SELECT 
            USER_NAME,
            WAREHOUSE_NAME,
            COUNT(*) AS QUERY_COUNT,
            SUM(CREDITS_ATTRIBUTED_COMPUTE) AS TOTAL_WAREHOUSE_CREDITS
        FROM SIMULATED_QUERY_ATTRIBUTION
        GROUP BY 1, 2
        ORDER BY TOTAL_WAREHOUSE_CREDITS DESC
    """).to_pandas()

print("=== Warehouse Compute Credits (Agent-Triggered Queries) ===")
print(df_compute.to_string(index=False))
print("\nTip: Set QUERY_TAG on sessions to attribute compute. Example:")
print("  ALTER SESSION SET QUERY_TAG = 'app=my_agent,team=analytics';")

=== Warehouse Compute Credits (Agent-Triggered Queries) ===
USER_NAME WAREHOUSE_NAME  QUERY_COUNT  TOTAL_WAREHOUSE_CREDITS
   SYSTEM     COMPUTE_WH         1920                 0.276966
PERICKSON     COMPUTE_WH          154                 0.106463

Tip: Set QUERY_TAG on sessions to attribute compute. Example:
  ALTER SESSION SET QUERY_TAG = 'app=my_agent,team=analytics';


---
## Section 6: Total Cost of a Request (Tokens + Compute)

A single agent request incurs multiple cost components:
1. **Token credits** (from `CORTEX_AGENT_USAGE_HISTORY.TOKEN_CREDITS`) -- the LLM inference cost
2. **AI function credits** (from `METADATA:ai_functions_credits`) -- any AI functions invoked
3. **Warehouse compute credits** (from `QUERY_ATTRIBUTION_HISTORY`) -- SQL execution cost

**Join key:** The agent automatically tags its generated SQL queries with `QUERY_TAG = 'cortex-agent-{REQUEST_ID}'`. This enables a precise join between agent requests and warehouse compute via:

```sql
QUERY_ATTRIBUTION_HISTORY.QUERY_TAG = 'cortex-agent-' || CORTEX_AGENT_USAGE_HISTORY.REQUEST_ID
```

**Note:** Queries under ~100ms may still show 0 compute (too short for attribution tracking).

In [9]:
# Total cost per request: tokens + AI functions + warehouse compute
#
# JOIN KEY: The agent automatically sets QUERY_TAG = 'cortex-agent-{REQUEST_ID}'
# on the SQL queries it generates. This enables a precise join to
# QUERY_ATTRIBUTION_HISTORY for per-request warehouse attribution.
#
# NOTE: Queries under ~100ms may still show 0 credits (too short for attribution).
# QUERY_ATTRIBUTION_HISTORY has up to 8-hour latency.

if USE_LIVE_DATA:
    df_total = session.sql("""
        SELECT
            a.REQUEST_ID,
            a.USER_NAME,
            a.AGENT_NAME,
            a.TOKEN_CREDITS AS ai_token_credits,
            COALESCE(a.METADATA:ai_functions_credits::NUMBER, 0) AS ai_fn_credits,
            COALESCE(SUM(q.CREDITS_ATTRIBUTED_COMPUTE), 0) AS warehouse_credits,
            a.TOKEN_CREDITS
                + COALESCE(a.METADATA:ai_functions_credits::NUMBER, 0)
                + COALESCE(SUM(q.CREDITS_ATTRIBUTED_COMPUTE), 0) AS total_credits
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY a
        LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY q
            ON q.QUERY_TAG = 'cortex-agent-' || a.REQUEST_ID
        WHERE a.START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
        GROUP BY 1, 2, 3, 4, 5
        ORDER BY total_credits DESC
        LIMIT 10
    """).to_pandas()

    print("=== Total Cost Per Request (Tokens + AI Functions + Warehouse) ===")
    print(df_total.to_string(index=False))
    print("""
JOIN: QUERY_ATTRIBUTION_HISTORY.QUERY_TAG = 'cortex-agent-' || REQUEST_ID
  - The agent automatically tags its generated SQL with this pattern
  - warehouse_credits = 0 means queries ran < ~100ms (too short for attribution)
  - QUERY_ATTRIBUTION_HISTORY has up to 8-hour data latency
""")
else:
    df_total = session.sql("""
        WITH agent_costs AS (
            SELECT 
                REQUEST_ID,
                USER_NAME,
                AGENT_NAME,
                SUM(TOKEN_CREDITS) AS AI_TOKEN_CREDITS,
                SUM(COALESCE(AI_FUNCTIONS_CREDITS, 0)) AS AI_FUNCTIONS_CREDITS
            FROM SIMULATED_AGENT_USAGE
            GROUP BY 1, 2, 3
        )
        SELECT 
            a.REQUEST_ID,
            a.USER_NAME,
            a.AGENT_NAME,
            a.AI_TOKEN_CREDITS,
            a.AI_FUNCTIONS_CREDITS,
            COALESCE(q.CREDITS_ATTRIBUTED_COMPUTE, 0) AS WAREHOUSE_CREDITS,
            a.AI_TOKEN_CREDITS + a.AI_FUNCTIONS_CREDITS + COALESCE(q.CREDITS_ATTRIBUTED_COMPUTE, 0) AS TOTAL_CREDITS
        FROM agent_costs a
        LEFT JOIN SIMULATED_QUERY_ATTRIBUTION q ON a.REQUEST_ID = q.QUERY_ID
        ORDER BY TOTAL_CREDITS DESC
        LIMIT 10
    """).to_pandas()
    print("=== Total Cost Per Request (Tokens + AI Functions + Warehouse) ===")
    print(df_total.to_string(index=False))

=== Total Cost Per Request (Tokens + AI Functions + Warehouse) ===
                          REQUEST_ID USER_NAME AGENT_NAME  AI_TOKEN_CREDITS  AI_FN_CREDITS  WAREHOUSE_CREDITS  TOTAL_CREDITS
862eeed0-16cc-43b8-91ad-59cfc7f47c55      None   SALESBRO          0.456023              0           0.000013       0.456037
bc0bf91e-1cc4-4526-a541-6088c4d8803b      None   SALESBRO          0.443833              0           0.000000       0.443833
df6420f3-a348-4445-bd12-2ac867db9bce      None   SALESBRO          0.393729              0           0.000000       0.393729
3bd5a293-6a8f-4392-859f-0f5d73b16e09      None   SALESBRO          0.380581              0           0.000055       0.380636
37e9d9f3-cde4-4eaf-a50c-55afbf685da6      None   SALESBRO          0.361677              0           0.000000       0.361677
f3983c5c-a02a-4ee2-94db-e686ed190b04      None   SALESBRO          0.348079              0           0.000000       0.348079
9b3027fc-660b-4fd0-852c-153867219b47      None   SALESBRO 

---
## Section 7: Daily Credit Trend and Service Type Reconciliation

At a higher level, `METERING_DAILY_HISTORY` shows total daily credits by `SERVICE_TYPE`:
- `CORTEX_AGENTS` = Agent orchestration token credits
- `AI_SERVICES` = Cortex Analyst, Search, AI Functions, Fine-tuning (rolled up)
- `WAREHOUSE_METERING` = All warehouse compute (not AI-specific)

Use this for executive-level reporting and to reconcile with billing.

In [10]:
# Daily credit trend by service type
if USE_LIVE_DATA:
    df_daily = session.sql("""
        SELECT 
            USAGE_DATE,
            SERVICE_TYPE,
            CREDITS_USED
        FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
        WHERE SERVICE_TYPE IN ('CORTEX_AGENTS', 'AI_SERVICES')
            AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
        ORDER BY USAGE_DATE DESC, SERVICE_TYPE
    """).to_pandas()
    print("=== Daily AI Credit Consumption (Last 30 Days) ===")
    print(df_daily.to_string(index=False))
else:
    print("METERING_DAILY_HISTORY is not available in simulated data.")
    print("\nIn production, query:")
    print("""
    SELECT 
        USAGE_DATE,
        SERVICE_TYPE,
        CREDITS_USED
    FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
    WHERE SERVICE_TYPE IN ('CORTEX_AGENTS', 'AI_SERVICES')
        AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
    ORDER BY USAGE_DATE DESC;
    """)
    print("SERVICE_TYPE mapping:")
    print("  CORTEX_AGENTS  -> Agent orchestration (token credits)")
    print("  AI_SERVICES    -> Cortex Analyst + Search + AI Functions + Fine-tuning")

=== Daily AI Credit Consumption (Last 30 Days) ===
USAGE_DATE  SERVICE_TYPE  CREDITS_USED
2026-07-21   AI_SERVICES      0.042300
2026-07-21 CORTEX_AGENTS      1.867431
2026-07-20 CORTEX_AGENTS     36.223465
2026-07-19 CORTEX_AGENTS     14.541327
2026-07-15 CORTEX_AGENTS      9.271696
2026-07-09 CORTEX_AGENTS      0.895504
2026-07-08 CORTEX_AGENTS      8.522893
2026-06-25 CORTEX_AGENTS      1.655280


---
## Section 8: Non-Snowflake User ID Attribution

In multi-tenant apps, the Snowflake `USER_NAME` is often a shared service account. To attribute costs to your application-level users, you have three options:

1. **Session Attributes** (`variables` in `agent:run` API) -- immutable per-session, readable via `SYS_CONTEXT()`
2. **USER_TAGS** -- Snowflake tags applied to users, visible in `CORTEX_AGENT_USAGE_HISTORY.USER_TAGS`
3. **QUERY_TAG** -- set on the session, visible in `QUERY_ATTRIBUTION_HISTORY.QUERY_TAG`

See: [Multi-tenancy for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-multi-tenancy)

In [11]:
# Tracking non-Snowflake user IDs with session attributes
#
# When building multi-tenant apps, the Snowflake USER_NAME is often a service
# account. To attribute costs to your *application-level* users, use the
# `variables` block in the agent:run API with `is_immutable_session_attribute: true`.
#
# These session attributes can be read via:
#   SYS_CONTEXT('SNOWFLAKE$SESSION_ATTRIBUTES', 'key')
#
# They appear in QUERY_HISTORY via QUERY_TAG if you set it in your app code.

print("""
=== Tracking Application-Level User IDs ===

Option 1: Session Attributes via agent:run API (multi-tenancy)
--------------------------------------------------------------
POST /api/v2/databases/{db}/schemas/{schema}/agents/{name}:run

{
    "variables": {
        "app_user_id": {
            "value": "user-12345",
            "type": "string",
            "is_immutable_session_attribute": true
        }
    },
    "messages": [...]
}

In SQL, read the attribute:
  SELECT SYS_CONTEXT('SNOWFLAKE$SESSION_ATTRIBUTES', 'app_user_id');

These attributes are immutable for the session duration and work with
row access policies for tenant isolation.

Docs: https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-multi-tenancy


Option 2: USER_TAGS in CORTEX_AGENT_USAGE_HISTORY
-------------------------------------------------
Apply Snowflake tags to users for cost-center grouping:

  ALTER USER my_service_account SET TAG my_db.tags.app_user = 'user-12345';

These appear in the USER_TAGS array column of CORTEX_AGENT_USAGE_HISTORY:
  [{
    "level": "USER",
    "tag_database": "MY_DB",
    "tag_schema": "TAGS", 
    "tag_name": "app_user",
    "tag_value": "user-12345"
  }]

Query by tag:
  SELECT
      t.value:tag_value::STRING AS app_user,
      SUM(TOKEN_CREDITS) AS credits
  FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY,
      LATERAL FLATTEN(input => USER_TAGS) t
  WHERE t.value:tag_name::STRING = 'app_user'
  GROUP BY 1
  ORDER BY credits DESC;


Option 3: QUERY_TAG for warehouse attribution
----------------------------------------------
Set QUERY_TAG on the session before agent queries execute:

  ALTER SESSION SET QUERY_TAG = 'app_user=user-12345,tenant=acme';

This makes the tag visible in QUERY_ATTRIBUTION_HISTORY.QUERY_TAG
for per-query warehouse cost attribution to app-level users.
""")


=== Tracking Application-Level User IDs ===

Option 1: Session Attributes via agent:run API (multi-tenancy)
--------------------------------------------------------------
POST /api/v2/databases/{db}/schemas/{schema}/agents/{name}:run

{
    "variables": {
        "app_user_id": {
            "value": "user-12345",
            "type": "string",
            "is_immutable_session_attribute": true
        }
    },
    "messages": [...]
}

In SQL, read the attribute:
  SELECT SYS_CONTEXT('SNOWFLAKE$SESSION_ATTRIBUTES', 'app_user_id');

These attributes are immutable for the session duration and work with
row access policies for tenant isolation.

Docs: https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-multi-tenancy


Option 2: USER_TAGS in CORTEX_AGENT_USAGE_HISTORY
-------------------------------------------------
Apply Snowflake tags to users for cost-center grouping:

  ALTER USER my_service_account SET TAG my_db.tags.app_user = 'user-12345';

These appear in the U

---
## Section 8: Resource Budgets (Governance)

Snowflake supports tag-based resource budgets for Cortex Agents. You can:
1. Create a tag and apply it to an agent
2. Create a budget with a monthly credit limit
3. Configure threshold actions (alerts at 80%, revoke access at 100%)

See: [Resource Budgets for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-resource-budgets)

In [12]:
# Resource Budget setup (reference -- do not run unless you want to create budget objects)
print("""
-- Step 1: Create a cost-center tag
CREATE TAG cost_mgmt_db.tags.cost_center
    ALLOWED_VALUES 'engineering', 'finance', 'analytics'
    COMMENT = 'Cost center tag for agent budgets';

-- Step 2: Apply the tag to your agent
ALTER AGENT IF EXISTS my_db.my_schema.my_agent
    SET TAG cost_mgmt_db.tags.cost_center = 'finance';

-- Step 3: Create a budget instance
USE SCHEMA budgets_db.budgets_schema;
CREATE SNOWFLAKE.CORE.BUDGET my_agent_budget();
CALL my_agent_budget!SET_SPENDING_LIMIT(500);  -- 500 credits/month

-- Step 4: Associate the tag with the budget
CALL my_agent_budget!SET_RESOURCE_TAGS(
    [[(SELECT SYSTEM$REFERENCE('TAG', 'cost_mgmt_db.tags.cost_center', 'SESSION', 'applybudget')), 'finance']],
    'UNION'
);

-- Step 5: Set notification at 80%
CALL my_agent_budget!SET_EMAIL_NOTIFICATIONS('budgets_integration', 'admin@company.com');
CALL my_agent_budget!SET_NOTIFICATION_THRESHOLD(80);

-- Step 6: Monitor usage
CALL my_agent_budget!GET_SERVICE_TYPE_USAGE_V2('2026-07', '2026-08');
""")


-- Step 1: Create a cost-center tag
CREATE TAG cost_mgmt_db.tags.cost_center
    ALLOWED_VALUES 'engineering', 'finance', 'analytics'
    COMMENT = 'Cost center tag for agent budgets';

-- Step 2: Apply the tag to your agent
ALTER AGENT IF EXISTS my_db.my_schema.my_agent
    SET TAG cost_mgmt_db.tags.cost_center = 'finance';

-- Step 3: Create a budget instance
USE SCHEMA budgets_db.budgets_schema;
CREATE SNOWFLAKE.CORE.BUDGET my_agent_budget();
CALL my_agent_budget!SET_SPENDING_LIMIT(500);  -- 500 credits/month

-- Step 4: Associate the tag with the budget
CALL my_agent_budget!SET_RESOURCE_TAGS(
    [[(SELECT SYSTEM$REFERENCE('TAG', 'cost_mgmt_db.tags.cost_center', 'SESSION', 'applybudget')), 'finance']],
    'UNION'
);

-- Step 5: Set notification at 80%
CALL my_agent_budget!SET_EMAIL_NOTIFICATIONS('budgets_integration', 'admin@company.com');
CALL my_agent_budget!SET_NOTIFICATION_THRESHOLD(80);

-- Step 6: Monitor usage
CALL my_agent_budget!GET_SERVICE_TYPE_USAGE_V2('2026-07', '2026

---
## Key Takeaways

| What | View | Granularity | Latency |
|------|------|-------------|---------|
| Token credits per request | `CORTEX_AGENT_USAGE_HISTORY` | Per-request | Up to 3 hours |
| Model-level token breakdown | `CORTEX_AGENT_USAGE_HISTORY.TOKENS_GRANULAR` | Per-sub-request | Up to 3 hours |
| Analyst credits (hourly) | `CORTEX_ANALYST_USAGE_HISTORY` | Hourly rollup | Up to 3 hours |
| Warehouse compute per query | `QUERY_ATTRIBUTION_HISTORY` | Per-query | Up to 8 hours |
| Daily totals by service | `METERING_DAILY_HISTORY` | Daily | Up to 3 hours |

**Important caveats:**
- ACCOUNT_USAGE views have up to 3-hour latency (sometimes longer)
- `QUERY_ATTRIBUTION_HISTORY` can take up to 8 hours
- Short queries (< ~100ms) may not appear in attribution views
- AI_SERVICES in metering is a roll-up of multiple feature views -- do not sum them together
- Token credits ≠ warehouse credits: they are separate billing dimensions

In [13]:
# Optional: Clean up lab objects (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS AGENT_COST_LAB").collect()
# session.sql("DROP WAREHOUSE IF EXISTS AGENT_COST_LAB_WH").collect()
print("Lab complete. Uncomment the lines above to clean up lab objects.")

Lab complete. Uncomment the lines above to clean up lab objects.
